# Econometría I — Tarea-examen

**Facultad de Ciencias, UNAM · Semestre 2027-I**
**Publicación:** martes 17 de noviembre de 2026 · **Entrega:** domingo 22 de noviembre de 2026, 23:59 h

Cubre todo el curso hasta ahora (Notas de Clase, caps. 1 a 7): esperanza condicional y DAG, MCO, variables
instrumentales, MCG, sistemas de ecuaciones y SUR, datos panel, máxima verosimilitud, respuesta binaria,
multinomial y ordenada, y modelos de conteo.

| Parte | Preguntas | Puntos |
|---|---|:--:|
| I. Teoría | T1 a T5 | 50 |
| II. Práctica | P1 a P5 | 50 |
| Bitácora de uso de IA | — | obligatoria |
| **Total** | | **100** |

### Instrucciones

1. Escriba su **número de cuenta** en la celda de abajo y corra el cuaderno completo (*Kernel → Restart & Run All*).
   Sus parámetros, sus datos y, por lo tanto, sus cifras son **sólo suyos**: el profesor los reproduce a partir de
   su número de cuenta. Un número copiado de otra persona no coincidirá con el suyo.
2. Responda en las celdas marcadas **Respuesta**. Las derivaciones pueden ir en LaTeX dentro de la celda o a mano:
   en ese caso, adjunte un PDF o inserte fotos legibles e indique en la celda dónde está cada derivación.
3. Las preguntas prácticas tienen cuatro partes, y **todas cuentan**:
   - **Predicción (2 pts).** *Antes* de correr nada, escriba qué espera y por qué. Se califica el razonamiento,
     no el acierto: una predicción equivocada y bien razonada vale los dos puntos.
   - **Resultado (4 pts).** Los números, obtenidos con su código y su muestra.
   - **Lectura (2 pts).** Qué significan, en unidades y en palabras, y si coinciden con su predicción.
   - **Verificación (2 pts).** Una comprobación independiente: una identidad que debe cumplirse, un cálculo a
     mano o un contraste con un resultado conocido.
4. **Entrega** en un solo correo dirigido a los tres profesores: `benjov@ciencias.unam.mx`,
   `jesicatapia@gmail.com` y `omarxalpha@gmail.com`. Asunto: `Tarea-examen Econometría I – Apellido Nombre`.
   Adjunte el cuaderno **ejecutado**, con el nombre `TE_Apellido_Nombre_Cuenta.ipynb`, y el PDF de las
   derivaciones a mano si lo hay.
5. Si por el paro, por trabajo o por otra razón no puede entregar el 22 de noviembre, escriba a los mismos tres
   correos a más tardar ese día: recibirá una versión equivalente con entrega el domingo 6 de diciembre.
6. Quien no presentó el primer parcial, o quien obtenga aquí una calificación mayor, tendrá esta tarea-examen como
   sustituto de esa calificación.

### Sobre el uso de inteligencia artificial

**Está permitido**, y la tarea está diseñada para que, aun usándola, usted aprenda. Por eso:

- Sus datos son personales: un asistente puede escribir el código, pero el número sólo sale de *su* corrida.
- Cada pregunta práctica pide una **predicción** antes de correr y una **verificación** después. Son las dos cosas
  que un asistente no puede hacer por usted: la primera exige que usted piense antes de ver el resultado, y la
  segunda exige que usted desconfíe de él.
- La pregunta T5 le pide **auditar** una respuesta escrita por un asistente de IA. Varias de sus afirmaciones son
  falsas y suenan convincentes.
- Al final hay una **bitácora** obligatoria de uso de IA. Usar IA no resta puntos; omitir la bitácora resta 5.

Una buena forma de usar un asistente: pídale que **le explique** un paso, no que le dé la respuesta; y antes de
aceptar cualquier cifra, compruébela con una de las verificaciones de la tarea. El sitio del curso tiene el
material de repaso: <https://benjov.github.io/Econometria-I-2026/repaso.html>.

In [ ]:
CUENTA = "000000000"     # <-- su número de cuenta, 9 dígitos, entre comillas

In [ ]:
# ---------------------------------------------------------------------------
# Personalización. NO MODIFIQUE ESTA CELDA.
# Todo lo que depende de su número de cuenta sale de aquí: los parámetros de
# las preguntas teóricas y las muestras de las prácticas. Así, sus cifras son
# sólo suyas y el profesor puede reproducirlas exactamente.
# ---------------------------------------------------------------------------
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

CONVOCATORIA = 1          # versión de la tarea-examen (no la cambie)

if not (isinstance(CUENTA, str) and CUENTA.isdigit() and len(CUENTA) == 9
        and CUENTA != "000000000"):
    raise ValueError("Escriba su número de cuenta (9 dígitos, entre comillas) "
                     "en la celda anterior y vuelva a correr el cuaderno.")


def generador(etiqueta):
    """Generador aleatorio propio de su cuenta y de cada pregunta."""
    clave = f"{CUENTA}-{CONVOCATORIA}-{etiqueta}"
    semilla = int(hashlib.sha256(clave.encode()).hexdigest()[:12], 16)
    return np.random.default_rng(semilla)


def _elige(g, lo, hi, paso):
    n = int(round((hi - lo) / paso))
    return round(lo + paso * int(g.integers(0, n + 1)), 4)


def mis_parametros():
    """Parámetros personales de las preguntas teóricas T1 a T4."""
    g = generador("teoria")
    s1 = 1 if g.integers(0, 2) else -1
    s2 = 1 if g.integers(0, 2) else -1
    return {
        # T1: y = b1*x + b2*z + u,  x = pi*z + v
        "T1_b1": _elige(g, 0.5, 1.5, 0.1),
        "T1_b2": s1 * _elige(g, 0.4, 1.2, 0.1),
        "T1_pi": s2 * _elige(g, 0.3, 1.0, 0.1),
        "T1_var_z": _elige(g, 1.0, 4.0, 0.5),
        "T1_var_v": _elige(g, 0.5, 2.0, 0.5),
        # T2: y = b*x + u,  x = pi*z + v,  Var(z) = Var(v) = 1
        "T2_pi": _elige(g, 0.05, 0.60, 0.05),
        "T2_cov_zu": _elige(g, 0.02, 0.10, 0.01),
        "T2_cov_vu": _elige(g, 0.30, 0.80, 0.10),
        # T3: datos panel con efectos aleatorios
        "T3_var_alfa": _elige(g, 0.2, 2.0, 0.1),
        "T3_var_eps": _elige(g, 0.5, 2.0, 0.1),
        "T3_M": int(g.integers(3, 11)),
        # T4: logit multinomial (base 0 = escuela, 1 = casa, 2 = trabajo)
        "T4_indice_casa": _elige(g, -1.0, 2.0, 0.25),
        "T4_indice_trabajo": _elige(g, 0.5, 3.5, 0.25),
        "T4_b_casa": -_elige(g, 0.40, 0.90, 0.05),
        "T4_b_trabajo": -_elige(g, 0.05, 0.45, 0.05),
        # T4: probit ordenado con cuatro categorías (0, 1, 2, 3)
        "T4_alfa1": -_elige(g, 0.5, 1.5, 0.1),
        "T4_alfa2": _elige(g, -0.3, 0.5, 0.1),
        "T4_alfa3": _elige(g, 0.9, 2.0, 0.1),
        "T4_xb": _elige(g, -1.0, 1.5, 0.1),
        "T4_bk": _elige(g, 0.3, 1.2, 0.1),
    }


def parametros_P2():
    """Diseño personal de la simulación de variables instrumentales (P2)."""
    g = generador("P2-diseno")
    signo = 1 if g.integers(0, 2) else -1
    return {"pi": _elige(g, 0.04, 0.40, 0.02),
            "rho": signo * _elige(g, 0.3, 0.8, 0.1),
            "beta": 1.0, "N": 400}


def simular_P2():
    """Datos de la pregunta P2: y = 1 + beta*x + u,  x = 0.5 + pi*z + v."""
    p = parametros_P2()
    g = generador("P2-datos")
    N = p["N"]
    z = g.standard_normal(N)
    e1 = g.standard_normal(N)
    e2 = g.standard_normal(N)
    u = e1
    v = p["rho"] * e1 + np.sqrt(1 - p["rho"] ** 2) * e2      # Corr(u, v) = rho
    x = 0.5 + p["pi"] * z + v
    y = 1.0 + p["beta"] * x + u
    return pd.DataFrame({"y": y, "x": x, "z": z})


def mi_muestra(n_total, n_elegidos, etiqueta):
    """Índices (posiciones) de su submuestra personal, en orden creciente."""
    g = generador(etiqueta)
    return np.sort(g.permutation(n_total)[:n_elegidos])


# Los datos se leen del repositorio si el cuaderno corre dentro de él; si no
# (por ejemplo, en Colab), se descargan de GitHub.
_RAIZ_WEB = "https://raw.githubusercontent.com/benjov/Econometria-I-2026/main/"


def leer(ruta):
    local = Path("..") / ruta
    fuente = local if local.exists() else _RAIZ_WEB + ruta.replace(" ", "%20")
    if ruta.lower().endswith(".dta"):
        return pd.read_stata(fuente)
    return pd.read_csv(fuente)


def datos_P1():
    d = leer("Clase_01_RegresionLineal/nerlove63.dta")
    return d.iloc[mi_muestra(len(d), 120, "P1")].reset_index(drop=True)


def datos_P3():
    d = leer("Clase_04_DatosPanel/wage_panel.csv")
    ids = np.sort(d["nr"].unique())
    elegidos = ids[mi_muestra(len(ids), 400, "P3")]
    return d[d["nr"].isin(elegidos)].reset_index(drop=True)


def datos_P4():
    d = leer("Clase_06_LogitMultinomial/keane.dta")
    d = d[d["year"] == 87].reset_index(drop=True)
    return d.iloc[mi_muestra(len(d), 1480, "P4")].reset_index(drop=True)


def datos_P5():
    d = leer("Clase_08_ModelosDeConteo/FERTIL2.DTA")
    cols = ["children", "educ", "age", "agesq", "urban", "electric", "tv"]
    d = d.dropna(subset=cols).reset_index(drop=True)
    return d.iloc[mi_muestra(len(d), 3500, "P5")].reset_index(drop=True)


print(f"Cuenta {CUENTA}, convocatoria {CONVOCATORIA}: personalización lista.")

In [ ]:
# Bibliotecas que usará en la tarea. En Colab, si falta linearmodels, corra primero:
#   !pip install linearmodels
import warnings
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

try:
    from linearmodels.iv import IV2SLS
    from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
except ImportError:
    print("Falta linearmodels: instálelo con  pip install linearmodels  y vuelva a correr.")

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# Algunas cifras (efectos aleatorios, Hausman) cambian ligeramente entre versiones;
# el profesor las reproduce con las mismas que usted.
import sys, scipy, statsmodels
try:
    import linearmodels
    _lm = linearmodels.__version__
except ImportError:
    _lm = "no instalado"
print(f"Python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · "
      f"statsmodels {statsmodels.__version__} · linearmodels {_lm} · scipy {scipy.__version__}")

### Sus parámetros personales

Las preguntas teóricas T1 a T4 usan los valores de esta tabla. Cópielos a sus respuestas tal como aparecen.

In [ ]:
pd.Series(mis_parametros(), name="valor").to_frame()

---
# Parte I. Teoría (50 puntos)

## T1. Variable omitida y DAG (10 puntos) — Notas, caps. 1 y 2

El modelo verdadero es
$$ y = \beta_1 x + \beta_2 z + u, \qquad x = \pi z + v, $$
donde $z$, $v$ y $u$ tienen media cero, son mutuamente independientes y $\mathbb{E}[u \mid x, z] = 0$. Sus valores
son `T1_b1` $=\beta_1$, `T1_b2` $=\beta_2$, `T1_pi` $=\pi$, `T1_var_z` $=Var(z)$ y `T1_var_v` $=Var(v)$.
Un investigador omite $z$ y regresa $y$ sólo sobre $x$ (sin constante, todo tiene media cero).

**(a) [4 pts]** Demuestre que $\text{plim}\, \tilde\beta_1 = \beta_1 + \beta_2 \delta$, donde
$\delta = Cov(x, z)/Var(x)$, y escriba $\delta$ en términos de $\pi$, $Var(z)$ y $Var(v)$. ¿Qué regresión
auxiliar estima $\delta$?

**(b) [2 pts]** Con sus parámetros calcule $Var(x)$, $\delta$, el sesgo asintótico y $\text{plim}\,\tilde\beta_1$.
¿El sesgo es hacia arriba o hacia abajo, y por qué tiene ese signo?

**(c) [2 pts]** Dibuje (o describa con flechas) el DAG de este modelo. ¿Qué papel tiene $z$: confusor, mediador o
colisionador? Si en cambio la estructura fuera $x \rightarrow z \rightarrow y$ (con $z = \gamma x + w$), ¿qué
estimaría la regresión corta, y sería la correcta para el efecto **total** de $x$?

**(d) [2 pts]** Suponga ahora que existe $c = x + y + e$, con $e$ independiente de todo lo demás. Usando el DAG,
explique por qué agregar $c$ a la regresión larga sesgaría $\beta_1$ aunque «se controle por más variables».

**Respuesta T1.**

*Escriba aquí.*

## T2. Un instrumento débil y casi exógeno (10 puntos) — Notas, cap. 3

$$ y = \beta x + u, \qquad x = \pi z + v, $$
con $Var(z) = Var(v) = 1$, $z$ independiente de $v$, $Cov(v, u) =$ `T2_cov_vu` y $Cov(z, u) =$ `T2_cov_zu`:
el instrumento viola **un poco** la exogeneidad. Su $\pi$ es `T2_pi`. Todo tiene media cero.

**(a) [3 pts]** Demuestre que
$$ \text{plim}\,\hat\beta_{VI} - \beta = \frac{Cov(z,u)}{\pi}, \qquad
   \text{plim}\,\hat\beta_{MCO} - \beta = \frac{\pi\, Cov(z,u) + Cov(v,u)}{\pi^2 + 1}. $$

**(b) [2 pts]** Calcule ambos sesgos con sus parámetros. En su caso, ¿cuál estimador tiene mayor sesgo asintótico?

**(c) [3 pts]** Demuestre que, con todas las cantidades positivas, VI tiene mayor sesgo asintótico que MCO si y
sólo si $\pi < Cov(z,u)/Cov(v,u)$. Calcule ese umbral con sus parámetros y ubique su $\pi$ respecto de él.

**(d) [2 pts]** En muestras grandes, la $F$ de la primera etapa es aproximadamente $1 + N\pi^2$ (con
$Var(z) = Var(v) = 1$). Calcúlela para $N = 500$ con su $\pi$. ¿Pasa la regla de $F > 10$? Explique qué relación
hay entre esa regla y el resultado de (c).

**Respuesta T2.**

*Escriba aquí.*

## T3. Sistemas de ecuaciones y datos panel (10 puntos) — Notas, caps. 4 y 5

**(a) [4 pts]** Un sistema SUR de dos ecuaciones, $\mathbf{y}_g = \mathbf{X}\boldsymbol\beta_g + \mathbf{u}_g$,
$g = 1, 2$, tiene **los mismos regresores** $\mathbf{X}$ ($N \times K$) en ambas, y
$Var(\mathbf{u}) = \boldsymbol\Sigma \otimes \mathbf{I}_N$ con $\boldsymbol\Sigma$ no diagonal. Demuestre que el
estimador de MCG del sistema apilado es idéntico a MCO ecuación por ecuación.
*Sugerencia:* escriba el sistema como $\mathbf{y} = (\mathbf{I}_2 \otimes \mathbf{X})\boldsymbol\beta + \mathbf{u}$
y use $(\mathbf{A}\otimes\mathbf{B})(\mathbf{C}\otimes\mathbf{D}) = \mathbf{AC}\otimes\mathbf{BD}$.

**(b) [3 pts]** En un panel con $M = 2$ periodos, $y_{it} = \mathbf{x}_{it}\boldsymbol\beta + \alpha_i +
\varepsilon_{it}$, demuestre que el estimador intragrupos y el de primeras diferencias son numéricamente idénticos.

**(c) [3 pts]** En el modelo de efectos aleatorios, con sus valores `T3_var_alfa` $=\sigma_\alpha^2$,
`T3_var_eps` $=\sigma_\varepsilon^2$ y `T3_M` $= M$, calcule
$\theta = 1 - \sqrt{\sigma_\varepsilon^2 / (\sigma_\varepsilon^2 + M\sigma_\alpha^2)}$ y la correlación intraclase
$\sigma_\alpha^2/(\sigma_\alpha^2+\sigma_\varepsilon^2)$. ¿Su estimador de efectos aleatorios se parece más a la
regresión agrupada o a intragrupos? ¿Qué le pasa a $\theta$ cuando $M \to \infty$, y por qué tiene sentido?

**Respuesta T3.**

*Escriba aquí.*

## T4. Máxima verosimilitud y modelos no lineales (10 puntos) — Notas, caps. 6 y 7

**(a) [3 pts] Poisson.** Con $\mathbb{E}[y \mid \mathbf{x}] = \exp(\mathbf{x}\boldsymbol\beta)$, escriba la
log-verosimilitud, el puntaje y el hessiano. Demuestre que la log-verosimilitud es globalmente cóncava y que, si
$\mathbf{x}$ incluye una constante, $\sum_i (y_i - \hat\mu_i) = 0$.

**(b) [4 pts] Logit multinomial.** Alternativas: 0 = escuela (base), 1 = casa, 2 = trabajo. Para una persona,
$\mathbf{x}\boldsymbol\beta_1 =$ `T4_indice_casa` y $\mathbf{x}\boldsymbol\beta_2 =$ `T4_indice_trabajo`; los
coeficientes de la educación son $\beta_{1,educ} =$ `T4_b_casa` y $\beta_{2,educ} =$ `T4_b_trabajo`.
1. Derive $\partial P_j / \partial x_k = P_j\,[\beta_{jk} - \sum_h P_h \beta_{hk}]$ (con $\beta_{0k} = 0$).
2. Calcule las tres probabilidades y los tres efectos marginales de la educación. ¿Suman cero? ¿Por qué?
3. ¿El efecto sobre $P(\text{trabajo})$ tiene el mismo signo que $\beta_{2,educ}$? Explique por qué sí o por qué no
   con sus números.
4. Agregue una alternativa 3 idéntica a «trabajo» (mismo índice). Calcule las cuatro probabilidades que predice el
   logit y el cociente $P(\text{casa})/P(\text{escuela})$ antes y después. ¿Qué propiedad ilustra esto y por qué
   es poco realista?

**(c) [3 pts] Probit ordenado.** Cuatro categorías (0 a 3), umbrales $\alpha_1, \alpha_2, \alpha_3 =$ `T4_alfa1`,
`T4_alfa2`, `T4_alfa3`, índice $\mathbf{x}\boldsymbol\beta =$ `T4_xb` y coeficiente $\beta_k =$ `T4_bk`.
Calcule $P(y = j)$ y $\partial P(y=j)/\partial x_k$ para $j = 0, \ldots, 3$, usando
$\partial P(y=j)/\partial x_k = \beta_k[\phi(\alpha_j - \mathbf{x}\boldsymbol\beta) - \phi(\alpha_{j+1} -
\mathbf{x}\boldsymbol\beta)]$ con $\alpha_0 = -\infty$ y $\alpha_4 = +\infty$. Compruebe que los efectos suman cero
e indique qué categorías intermedias suben y cuáles bajan, y por qué.

**Respuesta T4.**

*Escriba aquí.*

## T5. Auditoría de una respuesta de IA (10 puntos)

Un compañero le pidió a un asistente de IA que le ayudara a interpretar los resultados de una tarea parecida a esta.
El asistente respondió lo siguiente:

> 1. En un modelo Poisson, si los datos tienen sobredispersión, los coeficientes estimados están sesgados; por eso
>    hay que cambiar a un modelo binomial negativo.
> 2. Un coeficiente de −0.024 para `educ` en un Poisson del número de hijos significa que cada año de educación
>    reduce el número esperado de hijos en 0.024.
> 3. Si los errores estándar robustos de un Poisson son menores que los clásicos, eso es consistente con
>    subdispersión.
> 4. En el logit multinomial, como el coeficiente de `black` en la ecuación de trabajo (frente a escuela) es
>    positivo, ser afroamericano aumenta la probabilidad de trabajar.
> 5. Los efectos marginales de una variable sobre las probabilidades de todas las alternativas de un logit
>    multinomial suman cero.
> 6. La prueba de Hausman rechazó la hipótesis nula; por lo tanto, el estimador de efectos aleatorios es el adecuado.
> 7. Si la primera etapa tiene $F = 4$, el estimador de 2SLS es insesgado, aunque poco preciso.
> 8. En un probit ordenado con $\beta_k > 0$, aumentar $x_k$ reduce la probabilidad de la categoría más baja y
>    aumenta la de la más alta; en las categorías intermedias el efecto puede tener cualquier signo.

**(a) [8 pts]** Para cada afirmación, diga si es **correcta** o **incorrecta** y justifique en una a tres líneas.
Si es incorrecta, escriba la versión correcta.

**(b) [2 pts]** Escoja **una** de las afirmaciones incorrectas y describa cómo la refutaría usted mismo con los datos
de la Parte II —qué correría y qué número esperaría ver—, sin preguntarle a ningún asistente.

**Respuesta T5.**

*Escriba aquí.*

---
# Parte II. Práctica (50 puntos)

Recuerde la estructura de cada pregunta: **Predicción** (2), **Resultado** (4), **Lectura** (2) y
**Verificación** (2). Escriba la predicción *antes* de correr el código y no la cambie después: si resultó
equivocada, explíquelo en la lectura. Eso también se califica bien.

## P1. Economías de escala y MCG: Nerlove (1963) (10 puntos) — `Clase_01`

Su muestra son 120 de las 145 plantas de Nerlove, elegidas con su número de cuenta (`datos_P1()`). El modelo, con la
homogeneidad de grado uno en precios ya impuesta, es
$$ \ln(C/p_F) = \beta_0 + \beta_Q \ln Q + \beta_L \ln(p_L/p_F) + \beta_K \ln(p_K/p_F) + u . $$

**Predicción.** Si hay economías de escala, ¿$\beta_Q$ es mayor o menor que 1? ¿Qué espera para la generación
eléctrica de 1955? ¿Espera que la varianza del error sea mayor en las plantas chicas o en las grandes, y por qué?

**Resultado.** (i) Estime por MCO con errores estándar clásicos y robustos (HC1). (ii) Contraste
$H_0: \beta_Q = 1$ con el error robusto. Calcule los rendimientos a escala $r = 1/\hat\beta_Q$ y su error estándar
por el método delta, $ee(r) = ee(\hat\beta_Q)/\hat\beta_Q^2$. (iii) Aplique la prueba de Breusch-Pagan.
(iv) MCG factible: divida la muestra en quintiles de producción, estime la varianza de los residuales de MCO en cada
grupo y reestime por mínimos cuadrados ponderados con pesos $1/\hat\sigma^2_g$.

**Lectura.** Interprete $r$ en palabras. Compare $\hat\beta_Q$ y su error estándar entre MCO-robusto y MCG: ¿cuál
de los dos remedios corrige la inferencia y cuál la eficiencia?

**Verificación.** Calcule $\hat{\boldsymbol\beta} = (\mathbf{X'X})^{-1}\mathbf{X'y}$ con `numpy` y compárelo con
`statsmodels`. Compruebe además que $\mathbf{X'e} = \mathbf{0}$.

In [ ]:
d1 = datos_P1()
print(d1.shape)
d1.head()

In [ ]:
# Su código para P1

**Respuesta P1 (predicción, resultado, lectura y verificación).**

*Escriba aquí.*

## P2. Variables instrumentales con datos simulados (10 puntos) — `Clase_02`

`simular_P2()` genera $N = 400$ observaciones de $y = 1 + \beta x + u$, $x = 0.5 + \pi z + v$, con $\beta = 1$,
$z \sim N(0,1)$ y $Corr(u, v) = \rho$. Su $\pi$ y su $\rho$ están en `parametros_P2()`. Aquí, a diferencia de un
problema real, **usted conoce la verdad**: úsela.

**Predicción.** Con su $\rho$, ¿hacia dónde está sesgado MCO? Calcule su probabilidad límite,
$1 + \rho/(\pi^2 + 1)$. Con su $\pi$, ¿qué $F$ de primera etapa espera ($\approx 1 + N\pi^2$) y qué tan confiable
espera que sea 2SLS?

**Resultado.** (i) MCO. (ii) Primera etapa y su $F$. (iii) 2SLS con `IV2SLS`. (iv) Prueba de Durbin-Wu-Hausman por
función de control: agregue el residual de la primera etapa a la regresión de $y$ sobre $x$ y contraste su
coeficiente.

**Lectura.** Compare MCO y 2SLS con $\beta = 1$. Si su $F$ es menor que 10, explique qué está viendo; si es mayor,
explique por qué 2SLS funciona. ¿La prueba de Hausman detecta la endogeneidad que usted sabe que existe?

**Verificación.** Haga la segunda etapa «a mano»: regrese $y$ sobre $\hat x$ por MCO. Compruebe que el coeficiente
coincide con el de `IV2SLS` y que el error estándar **no**. Explique cuál es el correcto y por qué.

In [ ]:
print(parametros_P2())
d2 = simular_P2()
d2.describe()

In [ ]:
# Su código para P2

**Respuesta P2 (predicción, resultado, lectura y verificación).**

*Escriba aquí.*

## P3. El premio salarial sindical con datos panel (10 puntos) — `Clase_04`

Su muestra son 400 de los 545 hombres del panel NLSY 1980-1987 (`datos_P3()`), 8 años cada uno.

**Predicción.** ¿Espera que el premio sindical estimado con la regresión agrupada sea mayor o menor que el de
efectos fijos? ¿Qué característica no observada del trabajador explicaría la diferencia?

**Resultado.** Estime `lwage` sobre `union` y `married` con tres estimadores:
(i) agrupado por MCO, agregando `educ`, `black`, `hisp`, `exper`, `expersq` y efectos de año, con errores agrupados
por individuo; (ii) efectos aleatorios con los mismos regresores; (iii) efectos fijos con `union`, `married`,
`expersq`, efectos individuales y efectos de año, con errores agrupados. Reporte el $\theta$ de efectos aleatorios
y calcule la prueba de Hausman sobre los coeficientes de `union`, `married` y `expersq` (use las varianzas no
robustas de ambos modelos; ¿cuántos grados de libertad tiene?).

**Lectura.** Interprete el premio sindical de efectos fijos en porcentaje. ¿Qué estimador elige y por qué? ¿Por qué
`educ`, `black` y `hisp` desaparecen en efectos fijos, y por qué `exper` tampoco puede estimarse junto con los
efectos de año?

**Verificación.** Reproduzca el coeficiente de `union` de efectos fijos «a mano»: reste a cada variable (incluidas
las dicotómicas de año) su media por individuo con `groupby("nr")[...].transform("mean")` y estime por MCO.
¿Qué pasaría si usara `groupby("nr").mean()` en lugar de `transform`?

In [ ]:
d3 = datos_P3()
print(d3.shape, d3["nr"].nunique(), "individuos")
d3.head()

In [ ]:
# Su código para P3

**Respuesta P3 (predicción, resultado, lectura y verificación).**

*Escriba aquí.*

## P4. Escuela, casa o trabajo: Keane y Wolpin (1997) (10 puntos) — `Clase_06`

Su muestra son 1,480 de los hombres jóvenes observados en 1987 (`datos_P4()`). La variable `status` vale
1 = escuela, 2 = casa, 3 = trabajo. Un asistente de IA escribió el código de la celda siguiente para preparar los
datos y estimar el modelo. **Corre sin errores.**

**Predicción.** Antes de correrlo: ¿cuántas observaciones debería usar el modelo? (Revise los datos.) Y antes de
estimar: ¿espera que el coeficiente de `educ` en la ecuación de trabajo y su efecto marginal sobre
$P(\text{trabajo})$ tengan el mismo signo? ¿Qué tendría que ocurrir para que no?

**Resultado.** (i) Corra el código del asistente y reporte el número de observaciones y el coeficiente de `black` en
la ecuación de casa. Diga qué hace mal el código y corríjalo. (ii) Estime el logit multinomial corregido con
`educ`, `exper`, `expersq` y `black`, con escuela como base. (iii) Obtenga los efectos marginales promedio (AME).

**Lectura.** Identifique las variables cuyo coeficiente en la ecuación de trabajo tiene **signo opuesto** a su AME
sobre $P(\text{trabajo})$ y explique el mecanismo con la fórmula del efecto marginal.

**Verificación.** Calcule «a mano» el AME de `educ` sobre $P(\text{trabajo})$ a partir de las probabilidades
predichas y los coeficientes, y compárelo con `get_margeff()`. Compruebe que, para cada variable, los AME de las
tres alternativas suman cero.

In [ ]:
d4 = datos_P4()

# --- Código escrito por un asistente de IA (revíselo antes de confiar en él) ---
k = d4.copy()
k["estado"] = 0                          # 0 = escuela
k.loc[k["status"] == 2, "estado"] = 1    # 1 = casa
k.loc[k["status"] == 3, "estado"] = 2    # 2 = trabajo
modelo_ia = smf.mnlogit("estado ~ educ + exper + expersq + black", k).fit(disp=0)
print(modelo_ia.summary())

In [ ]:
# Su código para P4

**Respuesta P4 (predicción, resultado, lectura y verificación).**

*Escriba aquí.*

## P5. Número de hijos con un modelo Poisson (10 puntos) — `Clase_08`

Su muestra son 3,500 mujeres de Botswana de los datos FERTIL2 (`datos_P5()`). Modele `children` con `educ`, `age`,
`agesq`, `urban`, `electric` y `tv`.

**Predicción.** Calcule la media y la varianza de `children` (sin condicionar). ¿Qué sugieren sobre la dispersión?
¿Espera que los errores estándar robustos del Poisson resulten mayores o menores que los clásicos? Escríbalo antes de
estimar.

**Resultado.** (i) Estime el Poisson con errores clásicos y robustos (`cov_type="HC0"`). (ii) Calcule la dispersión
de Pearson, $\frac{1}{N-K}\sum_i (y_i - \hat\mu_i)^2/\hat\mu_i$. (iii) Interprete `educ` como semielasticidad,
aproximada ($100\hat\beta$) y exacta ($100(e^{\hat\beta}-1)$). (iv) Calcule el efecto porcentual de un año más de
edad a los 25 y a los 40 años, y la edad en que el número esperado de hijos es máximo.

**Lectura.** ¿Coincide la dispersión de Pearson con lo que sugerían la media y la varianza sin condicionar? Explique
la diferencia. ¿Qué errores estándar reporta y por qué?

**Verificación.** Programe Newton-Raphson para el Poisson (unas diez líneas con `numpy`) y compruebe que reproduce los
coeficientes de `statsmodels`. Compruebe que $\sum_i (y_i - \hat\mu_i) = 0$ y que, por lo tanto, el AME de `educ`
es $\hat\beta_{educ}\,\bar y$; compárelo con `get_margeff()`.

In [ ]:
d5 = datos_P5()
print(d5.shape)
d5[["children", "educ", "age", "urban", "electric", "tv"]].describe()

In [ ]:
# Su código para P5

**Respuesta P5 (predicción, resultado, lectura y verificación).**

*Escriba aquí.*

---
# Bitácora de uso de IA (obligatoria)

Usar IA no resta puntos. Omitir esta bitácora resta 5. Sea concreto: una bitácora genérica («la usé para revisar»)
cuenta como omitida.

| | Respuesta |
|---|---|
| ¿Qué herramientas usó (o «ninguna»)? | |
| ¿En qué preguntas y para qué (escribir código, explicar un concepto, depurar, redactar)? | |
| Copie **un** prompt que le haya sido útil. | |
| Describa **una** ocasión en que la respuesta del asistente estuvo mal o incompleta, y cómo lo detectó. Si no ocurrió, diga cómo comprobó que sus respuestas eran correctas. | |
| ¿Qué aprendió que no sabía antes de empezar? | |

**Declaración.** Escriba su nombre como constancia de que las respuestas son suyas, de que entiende cada una y de que
puede explicarlas: *nombre aquí*.